In [4]:
import json
from pathlib import Path

import numpy as np
import lmfit

try:
    HERE = Path(__file__).resolve().parent      # running as a .py script
except NameError:
    HERE = Path(".").resolve()                  # running in a notebook in the trios data folder

# Same files as trios_analysis.ipynb
POLYMERS = {
    "FEP": {
        "ref": 330,
        "shift_file": "FEP/FEP 9835 2007390112 mfr 24.8 330°C.txt",
        "sweeps": {
            330: "FEP/FEP 9835 2007390112 mfr 24.8 330°C.txt",
            350: "FEP/TE9835 Lot 2007390112 at 350°C.txt",
            370: "FEP/TE9835 Lot 2007390112 at 370°C.txt",
        },
    },
    "PFA 440HPAX": {
        "ref": 330,
        "shift_file": "PFA/PFA 440HPAX Lot 2103470009 at 330°C.txt",
        "sweeps": {
            330: "PFA/PFA 440HPAX Lot 2103470009 at 330°C.txt",
            350: "PFA/PFA 440  Lot 2103470009 mfr 15.5 at 350°C.txt",
            370: "PFA/PFA 440  Lot 2103470009 mfr 15.5 at 370°C.txt",
        },
    },
    "HT 2181": {
        "ref": 300,
        "shift_file": "HT/HT 2181 1212PASK 22 MFR 6,4 315°C.txt",
        "sweeps": {
            285: "HT/HT 2181 1212PASK 22 MFR 6,4 285°C.txt",
            300: "HT/HT 2181 1212PASK 22 MFR 6,4 300°C.txt",
            315: "HT/HT 2181 1212PASK 22 MFR 6,4 315°C.txt",
        },
    },
}

def _num(x):
    try:
        return float(x)
    except ValueError:
        return np.nan


def read_steps(path):
    """Split a TRIOS export into its steps, keeping the variable names and data rows."""
    steps, cur = [], None
    for line in Path(path).read_text(encoding="utf-8", errors="replace").splitlines():
        if line.startswith("Step name\t"):
            cur = {"name": line.split("\t", 1)[1], "vars": None, "rows": []}
            steps.append(cur)
        elif cur is None:
            continue
        elif line.startswith("Variables\t"):
            cur["vars"] = line.split("\t")[1:]
        elif line.startswith("Data point\t"):
            cur["rows"].append([_num(x) for x in line.split("\t")[1:]])
    return steps


def column(step, name):
    idx = step["vars"].index(name)
    return np.array([r[idx] for r in step["rows"]], float)


def frequency_step(path):
    return next(s for s in read_steps(path) if s["name"].startswith("Frequency"))


def load_shift_table(path):
    step = next(s for s in read_steps(path) if "shift" in s["name"].lower())
    T, aT = column(step, "Temperature"), column(step, "aT (x variable)")
    _, idx = np.unique(np.round(T), return_index=True)  # drops the duplicated 300 °C row in the HT table
    return T[idx], aT[idx]


def log_cross(lx, leta0, llam, m):
    """Cross model (eta_inf = 0) in log form, same as the notebook."""
    return np.log(np.exp(leta0) / (1 + (np.exp(llam) * np.exp(lx)) ** m))

out = {}
for name, cfg in POLYMERS.items():
    Tt, aTt = load_shift_table(HERE / cfg["shift_file"])
    xs, etas = [], []
    for T, f in sorted(cfg["sweeps"].items()):
        step = frequency_step(HERE / f)
        w = column(step, "Angular frequency")
        eta = column(step, "Complex viscosity")
        aT = aTt[np.argmin(np.abs(Tt - T))]
        eta = eta * ((cfg["ref"] + 273.15) / (T + 273.15)) / aT   # same vertical shift as the notebook
        # same excluded low-frequency HT points as the notebook
        if name == "HT 2181" and T == 315:
            drop = np.argsort(w)[:3]
        elif name == "HT 2181" and T == 300:
            drop = np.argsort(w)[:1]
        else:
            drop = np.array([], int)
        w, eta = np.delete(w, drop), np.delete(eta, drop)
        xs.append(w * aT)
        etas.append(eta)

    x, eta = np.concatenate(xs), np.concatenate(etas)
    ok = (x > 0) & (eta > 0) & np.isfinite(eta)
    x, eta = x[ok], eta[ok]

    model = lmfit.Model(log_cross, independent_vars=["lx"])
    params = model.make_params(leta0=np.log(eta.max()), llam=np.log(1 / np.median(x)), m=0.8)
    params["m"].set(min=0.05, max=1.5)
    result = model.fit(np.log(eta), params, lx=np.log(x))

    names = result.var_names                                   # ['leta0', 'llam', 'm']
    best = {k: result.params[k].value for k in names}
    cov = np.array(result.covar)
    out[name] = {"covar": cov.tolist(), "vars": names, "best": best}

    # Print the fit and the correlation matrix so you can see how the parameters move together
    sd = np.sqrt(np.diag(cov))
    corr = cov / np.outer(sd, sd)
    print(f"===== {name} =====")
    print(f"  eta0 = {np.exp(best['leta0']):.4g} Pa s, lambda = {np.exp(best['llam']):.4g} s, n (m) = {best['m']:.4g}")
    print("  correlation matrix (ln eta0, ln lambda, n):")
    for i, n in enumerate(["ln eta0", "ln lam", "n"]):
        print(f"    {n:>8} " + " ".join(f"{corr[i, j]:6.2f}" for j in range(3)))
    print()

OUT = HERE.parent / "optimization" / "cross_covariance.json"
OUT.parent.mkdir(exist_ok=True)
with open(OUT, "w") as f:
    json.dump(out, f, indent=1)
print(f"Saved covariance matrices to {OUT}")
with open(OUT, "w") as f:
    json.dump(out, f, indent=1)
print(f"Saved covariance matrices to {OUT}")


===== FEP =====
  eta0 = 3040 Pa s, lambda = 0.01027 s, n (m) = 0.6861
  correlation matrix (ln eta0, ln lambda, n):
     ln eta0   1.00   0.80  -0.61
      ln lam   0.80   1.00  -0.67
           n  -0.61  -0.67   1.00

===== PFA 440HPAX =====
  eta0 = 4214 Pa s, lambda = 0.01317 s, n (m) = 0.692
  correlation matrix (ln eta0, ln lambda, n):
     ln eta0   1.00   0.80  -0.60
      ln lam   0.80   1.00  -0.72
           n  -0.60  -0.72   1.00

===== HT 2181 =====
  eta0 = 1.278e+04 Pa s, lambda = 1.034 s, n (m) = 0.5294
  correlation matrix (ln eta0, ln lambda, n):
     ln eta0   1.00   0.95  -0.78
      ln lam   0.95   1.00  -0.91
           n  -0.78  -0.91   1.00

Saved covariance matrices to /Users/britneysedano/Desktop/rheo/polymer/667-polymer-project/optimization/cross_covariance.json
Saved covariance matrices to /Users/britneysedano/Desktop/rheo/polymer/667-polymer-project/optimization/cross_covariance.json
